# Evaluate the Fixed Lambda Sweep

This notebook runs Step 13 of the thesis prototype. It evaluates the fixed lambda-grid generations stored in:

`results/adapter_merge_generations.csv`

The evaluation uses simple heuristic proxy scores for helpfulness and harmlessness. These are placeholder prototype scores, not final reward-model or RLHF evaluation.

This notebook does **not** train adapters, compute the relationship matrix $R$, or implement the final mapping $\lambda = f(p, R)$.

## 1. Clone or update the repository

This cell always begins in `/content`. If the repository already exists, it updates it with `git pull`. Otherwise, it clones the repository. This avoids nested folders such as `/content/master-thesis/master-thesis`.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if (repo_path / ".git").is_dir():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

## 2. Inspect the repository

You should see folders such as `scripts/`, `src/`, `notebooks/`, and `results/`. The `scripts/` folder should contain `evaluate_lambda_sweep.py`.

In [ ]:
!pwd
!ls
!ls scripts
!ls results || echo "No results folder found yet."

## 3. Install dependencies

The lambda-sweep evaluator itself uses only standard Python. Pandas is installed here for convenient CSV previews. No model libraries are needed for Step 13.

In [ ]:
!pip install -q pandas

## 4. Check the Step 12 input file

The required input is `results/adapter_merge_generations.csv`.

If it is missing, Step 12 must be completed first. Make sure the Helpful and Harmless adapters exist locally, then run:

```bash
python scripts/evaluate_adapter_merges.py
```

That script creates the fixed lambda-grid generations used by this notebook.

In [ ]:
from pathlib import Path

input_path = Path("results/adapter_merge_generations.csv")

if input_path.is_file():
    print(f"Input file found: {input_path}")
else:
    print("Input file is missing.")
    print("Run Step 12 first: python scripts/evaluate_adapter_merges.py")

## 5. Preview the input CSV

Continue only after the previous cell confirms that the input file exists. The first preview shows the raw CSV lines; the second displays a small table with pandas.

In [ ]:
!head -n 5 results/adapter_merge_generations.csv

In [ ]:
import pandas as pd

generations_df = pd.read_csv("results/adapter_merge_generations.csv")
print("Rows:", len(generations_df))
generations_df.head()

## 6. Evaluate the fixed lambda sweep

This script adds simple response-level heuristic proxies, aggregates them by lambda pair, and computes preference-weighted utilities for three example preference vectors.

It does not compute $R$ and does not implement $\lambda = f(p, R)$.

In [ ]:
!python scripts/evaluate_lambda_sweep.py

## 7. Inspect the evaluation outputs

The evaluator creates two small files:

- `results/adapter_merge_scored_generations.csv`: one row per generated response with proxy scores.
- `results/lambda_sweep_summary.csv`: one aggregate row per lambda pair.

In [ ]:
!ls results
!cat results/lambda_sweep_summary.csv

In [ ]:
scored_df = pd.read_csv("results/adapter_merge_scored_generations.csv")
summary_df = pd.read_csv("results/lambda_sweep_summary.csv")

print("Scored generations:")
display(scored_df.head())

print("Lambda-sweep summary:")
display(summary_df)

## 8. Git safety check

It is okay to commit the small CSV result files under `results/` if you want them in the repository.

Do **not** commit:

- `adapters/`
- `adapters.zip`
- `.safetensors` or `.bin` files
- checkpoints or full model files

These generated model artifacts should remain local.

In [ ]:
!git status

## What this notebook establishes

This notebook provides a first descriptive comparison of the fixed lambda grid using lightweight heuristic proxy scores. The proxies are not final objective scores. Computing adapter geometry $\delta_i$, constructing $R$, and implementing the final preference-aware correction $\lambda = f(p, R)$ remain later thesis steps.